## 데이터 파악

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

train = pd.read_csv('/content/drive/MyDrive/data/뉴스토픽 분류/train_data.csv')
test = pd.read_csv('/content/drive/MyDrive/data/뉴스토픽 분류/test_data.csv')
sample_submission = pd.read_csv('/content/drive/MyDrive/data/뉴스토픽 분류/sample_submission.csv')

In [ ]:
train.head()

,index,title,topic_idx
0,0,인천→핀란드 항공기 결항…휴가철 여행객 분통,4
1,1,실리콘밸리 넘어서겠다…구글 15조원 들여 美전역 거점화,4
2,2,이란 외무 긴장완화 해결책은 미국이 경제전쟁 멈추는 것,4
3,3,NYT 클린턴 측근韓기업 특수관계 조명…공과 사 맞물려종합,4
4,4,시진핑 트럼프에 중미 무역협상 조속 타결 희망,4


In [ ]:
train.shape

(45654, 3)

In [ ]:
test.head()

,index,title
0,45654,유튜브 내달 2일까지 크리에이터 지원 공간 운영
1,45655,어버이날 맑다가 흐려져…남부지방 옅은 황사
2,45656,내년부터 국가RD 평가 때 논문건수는 반영 않는다
3,45657,김명자 신임 과총 회장 원로와 젊은 과학자 지혜 모을 것
4,45658,회색인간 작가 김동식 양심고백 등 새 소설집 2권 출간


In [ ]:
test.shape

(9131, 2)

In [ ]:
sample_submission.head()

,index,topic_idx
0,45654,0
1,45655,0
2,45656,0
3,45657,0
4,45658,0


In [ ]:
train.isnull().sum()

,0
index,0
title,0
topic_idx,0


In [ ]:
test.isnull().sum()

,0
index,0
title,0


In [ ]:
train.title

,title
0,인천→핀란드 항공기 결항…휴가철 여행객 분통
1,실리콘밸리 넘어서겠다…구글 15조원 들여 美전역 거점화
2,이란 외무 긴장완화 해결책은 미국이 경제전쟁 멈추는 것
3,NYT 클린턴 측근韓기업 특수관계 조명…공과 사 맞물려종합
4,시진핑 트럼프에 중미 무역협상 조속 타결 희망
...,...
45649,KB금융 미국 IB 스티펠과 제휴…선진국 시장 공략
45650,1보 서울시교육청 신종코로나 확산에 개학 연기·휴업 검토
45651,게시판 키움증권 2020 키움 영웅전 실전투자대회
45652,답변하는 배기동 국립중앙박물관장


In [ ]:
train.topic_idx.value_counts()

,count
topic_idx,
4,7629
2,7362
5,6933
6,6751
1,6222
3,5933
0,4824


## 전처리

### 텍스트 정제

In [ ]:
import re

def clean_text(text):
  text = re.sub(r'[“”‘’]', '', text)
  text = re.sub(r'[^가-힣A-Za-z0-9\s]', ' ', text) # 한글, 영문, 숫자, 공백만 남김
  text = re.sub(r'\s+', ' ', text).strip() # 중복공백 제거
  return text

train['cleaned'] = train['title'].apply(clean_text)
test['cleaned'] = test['title'].apply(clean_text)
print('----train----')
print(train[['title', 'cleaned']])
print('----test----')
print(test[['title', 'cleaned']])

----train----
                                    title                             cleaned
0                인천→핀란드 항공기 결항…휴가철 여행객 분통            인천 핀란드 항공기 결항 휴가철 여행객 분통
1          실리콘밸리 넘어서겠다…구글 15조원 들여 美전역 거점화       실리콘밸리 넘어서겠다 구글 15조원 들여 전역 거점화
2          이란 외무 긴장완화 해결책은 미국이 경제전쟁 멈추는 것      이란 외무 긴장완화 해결책은 미국이 경제전쟁 멈추는 것
3        NYT 클린턴 측근韓기업 특수관계 조명…공과 사 맞물려종합    NYT 클린턴 측근 기업 특수관계 조명 공과 사 맞물려종합
4               시진핑 트럼프에 중미 무역협상 조속 타결 희망           시진핑 트럼프에 중미 무역협상 조속 타결 희망
...                                   ...                                 ...
45649        KB금융 미국 IB 스티펠과 제휴…선진국 시장 공략        KB금융 미국 IB 스티펠과 제휴 선진국 시장 공략
45650     1보 서울시교육청 신종코로나 확산에 개학 연기·휴업 검토     1보 서울시교육청 신종코로나 확산에 개학 연기 휴업 검토
45651         게시판 키움증권 2020 키움 영웅전 실전투자대회         게시판 키움증권 2020 키움 영웅전 실전투자대회
45652                   답변하는 배기동 국립중앙박물관장                   답변하는 배기동 국립중앙박물관장
45653  2020 한국인터넷기자상 시상식 내달 1일 개최…특별상 김성후  2020 한국인터넷기자상 시상식 내달 1일 개최 특별상 김성후

[45654 rows x 2 columns]
----test----
           

### 형태소 분석

In [ ]:
!pip install kiwipiepy

In [ ]:
from kiwipiepy import Kiwi
kiwi = Kiwi()

def kiwi_tokenize(text):
    return [t.form for t in kiwi.tokenize(text)
            if len(t.form) > 1 and t.tag.startswith(('N', 'V', 'M'))]

train['tokens'] = train['cleaned'].apply(kiwi_tokenize)
train['processed'] = train['tokens'].apply(lambda x: ' '.join(x))

test['tokens'] = test['cleaned'].apply(kiwi_tokenize)
test['processed'] = test['tokens'].apply(lambda x: ' '.join(x))

In [ ]:
train.head()

,index,title,topic_idx,cleaned,tokens,processed
0,0,인천→핀란드 항공기 결항…휴가철 여행객 분통,4,인천 핀란드 항공기 결항 휴가철 여행객 분통,"[인천, 핀란드, 항공기, 결항, 휴가철, 여행객, 분통]",인천 핀란드 항공기 결항 휴가철 여행객 분통
1,1,실리콘밸리 넘어서겠다…구글 15조원 들여 美전역 거점화,4,실리콘밸리 넘어서겠다 구글 15조원 들여 전역 거점화,"[실리콘밸리, 넘어서, 구글, 들이, 전역, 거점]",실리콘밸리 넘어서 구글 들이 전역 거점
2,2,이란 외무 긴장완화 해결책은 미국이 경제전쟁 멈추는 것,4,이란 외무 긴장완화 해결책은 미국이 경제전쟁 멈추는 것,"[이란, 외무, 긴장, 완화, 해결책, 미국, 경제, 전쟁, 멈추]",이란 외무 긴장 완화 해결책 미국 경제 전쟁 멈추
3,3,NYT 클린턴 측근韓기업 특수관계 조명…공과 사 맞물려종합,4,NYT 클린턴 측근 기업 특수관계 조명 공과 사 맞물려종합,"[클린턴, 측근, 기업, 특수, 관계, 조명, 공과, 맞물리, 종합]",클린턴 측근 기업 특수 관계 조명 공과 맞물리 종합
4,4,시진핑 트럼프에 중미 무역협상 조속 타결 희망,4,시진핑 트럼프에 중미 무역협상 조속 타결 희망,"[시진핑, 트럼프, 무역, 협상, 타결, 희망]",시진핑 트럼프 무역 협상 타결 희망


In [ ]:
test.head()

,index,title,cleaned,tokens,processed
0,45654,유튜브 내달 2일까지 크리에이터 지원 공간 운영,유튜브 내달 2일까지 크리에이터 지원 공간 운영,"[유튜브, 크리에이터, 지원, 공간, 운영]",유튜브 크리에이터 지원 공간 운영
1,45655,어버이날 맑다가 흐려져…남부지방 옅은 황사,어버이날 맑다가 흐려져 남부지방 옅은 황사,"[어버이날, 흐리, 남부, 지방, 황사]",어버이날 흐리 남부 지방 황사
2,45656,내년부터 국가RD 평가 때 논문건수는 반영 않는다,내년부터 국가RD 평가 때 논문건수는 반영 않는다,"[내년, 국가, 평가, 논문, 건수, 반영]",내년 국가 평가 논문 건수 반영
3,45657,김명자 신임 과총 회장 원로와 젊은 과학자 지혜 모을 것,김명자 신임 과총 회장 원로와 젊은 과학자 지혜 모을 것,"[김명자, 신임, 과총, 회장, 원로, 과학자, 지혜, 모으]",김명자 신임 과총 회장 원로 과학자 지혜 모으
4,45658,회색인간 작가 김동식 양심고백 등 새 소설집 2권 출간,회색인간 작가 김동식 양심고백 등 새 소설집 2권 출간,"[회색, 인간, 작가, 양심, 고백, 소설집, 출간]",회색 인간 작가 양심 고백 소설집 출간


### stopwords

In [ ]:
stopwords = set([
    '으로', '에서', '하다', '했다', '한다', '그리고', '그러나', '등',
    '대한', '관련', '위해', '통해', '대해', '것', '수', '이번', '지난', '지난해'
])

def remove_stopwords(tokens):
    return [t for t in tokens if t not in stopwords]

train['tokens'] = train['tokens'].apply(remove_stopwords)
train['processed'] = train['tokens'].apply(lambda x: ' '.join(x))

test['tokens'] = test['tokens'].apply(remove_stopwords)
test['processed'] = test['tokens'].apply(lambda x: ' '.join(x))

In [ ]:
train.head()

,index,title,topic_idx,cleaned,tokens,processed
0,0,인천→핀란드 항공기 결항…휴가철 여행객 분통,4,인천 핀란드 항공기 결항 휴가철 여행객 분통,"[인천, 핀란드, 항공기, 결항, 휴가철, 여행객, 분통]",인천 핀란드 항공기 결항 휴가철 여행객 분통
1,1,실리콘밸리 넘어서겠다…구글 15조원 들여 美전역 거점화,4,실리콘밸리 넘어서겠다 구글 15조원 들여 전역 거점화,"[실리콘밸리, 넘어서, 구글, 들이, 전역, 거점]",실리콘밸리 넘어서 구글 들이 전역 거점
2,2,이란 외무 긴장완화 해결책은 미국이 경제전쟁 멈추는 것,4,이란 외무 긴장완화 해결책은 미국이 경제전쟁 멈추는 것,"[이란, 외무, 긴장, 완화, 해결책, 미국, 경제, 전쟁, 멈추]",이란 외무 긴장 완화 해결책 미국 경제 전쟁 멈추
3,3,NYT 클린턴 측근韓기업 특수관계 조명…공과 사 맞물려종합,4,NYT 클린턴 측근 기업 특수관계 조명 공과 사 맞물려종합,"[클린턴, 측근, 기업, 특수, 관계, 조명, 공과, 맞물리, 종합]",클린턴 측근 기업 특수 관계 조명 공과 맞물리 종합
4,4,시진핑 트럼프에 중미 무역협상 조속 타결 희망,4,시진핑 트럼프에 중미 무역협상 조속 타결 희망,"[시진핑, 트럼프, 무역, 협상, 타결, 희망]",시진핑 트럼프 무역 협상 타결 희망


In [ ]:
test.head()

,index,title,cleaned,tokens,processed
0,45654,유튜브 내달 2일까지 크리에이터 지원 공간 운영,유튜브 내달 2일까지 크리에이터 지원 공간 운영,"[유튜브, 크리에이터, 지원, 공간, 운영]",유튜브 크리에이터 지원 공간 운영
1,45655,어버이날 맑다가 흐려져…남부지방 옅은 황사,어버이날 맑다가 흐려져 남부지방 옅은 황사,"[어버이날, 흐리, 남부, 지방, 황사]",어버이날 흐리 남부 지방 황사
2,45656,내년부터 국가RD 평가 때 논문건수는 반영 않는다,내년부터 국가RD 평가 때 논문건수는 반영 않는다,"[내년, 국가, 평가, 논문, 건수, 반영]",내년 국가 평가 논문 건수 반영
3,45657,김명자 신임 과총 회장 원로와 젊은 과학자 지혜 모을 것,김명자 신임 과총 회장 원로와 젊은 과학자 지혜 모을 것,"[김명자, 신임, 과총, 회장, 원로, 과학자, 지혜, 모으]",김명자 신임 과총 회장 원로 과학자 지혜 모으
4,45658,회색인간 작가 김동식 양심고백 등 새 소설집 2권 출간,회색인간 작가 김동식 양심고백 등 새 소설집 2권 출간,"[회색, 인간, 작가, 양심, 고백, 소설집, 출간]",회색 인간 작가 양심 고백 소설집 출간


### TF-IDF 벡터화

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features = 15000,
    ngram_range = (1,3),
    min_df = 2,
    max_df = 0.95,
    sublinear_tf = True
)

X_train = vectorizer.fit_transform(train['processed'])
X_test = vectorizer.transform(test['processed'])
y_train = train['topic_idx']

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (45654, 15000)
X_test shape: (9131, 15000)


## 모델링 - RandomForestClassifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

pred = rf.predict(X_test)

In [ ]:
sample_submission['topic_idx'] = pred
sample_submission.to_csv('뉴스토픽_결과.csv', index=False)
print("제출 파일 저장 완료!")

제출 파일 저장 완료!


## 스태킹

In [ ]:
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, StackingClassifier
from lightgbm import LGBMClassifier
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
import numpy as np

# 베이스 모델 2~3개 구성 (랜덤포레스트 기반)
base_estimators = [
    ('rf1', RandomForestClassifier(
        n_estimators=300, max_depth=30, min_samples_split=3,
        min_samples_leaf=1, max_features='sqrt', random_state=42, n_jobs=1)),

    ('rf2', RandomForestClassifier(
        n_estimators=400, max_depth=None, min_samples_split=5,
        min_samples_leaf=2, max_features='log2', random_state=43, n_jobs=1))
]



# 메타모델: LightGBM
meta_model = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=-1,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=1
)

# 파이프라인 구성
stack_pipeline = Pipeline([
    ('select', SelectKBest(chi2, k=12000)),
    ('stack', StackingClassifier(
        estimators=base_estimators,
        final_estimator=meta_model,
        cv=3,
        n_jobs=1,
        passthrough=True
    ))
])

# 전체 데이터로 학습 & 예측
stack_pipeline.fit(X_train, y_train)
stack_pred = stack_pipeline.predict(X_test)

# 제출 파일 생성
sample_submission['topic_idx'] = stack_pred
sample_submission.to_csv('/content/drive/MyDrive/data/뉴스토픽_스태킹결과1.csv', index=False)
print("제출 파일 저장 완료!")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 1.404754 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 82192
[LightGBM] [Info] Number of data points in the train set: 45654, number of used features: 3123
[LightGBM] [Info] Start training from score -2.247488
[LightGBM] [Info] Start training from score -1.993000
[LightGBM] [Info] Start training from score -1.824760
[LightGBM] [Info] Start training from score -2.040561
[LightGBM] [Info] Start training from score -1.789134
[LightGBM] [Info] Start training from score -1.884799
[LightGBM] [Info] Start training from score -1.911401


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


제출 파일 저장 완료!


## 스태킹 최적 조합 찾기

In [ ]:
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier, StackingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.pipeline import Pipeline
import numpy as np

# 후보 base/meta 세트 (소형)
base_model_sets = {
    'rf_et': [
        ('rf', RandomForestClassifier(n_estimators=150, max_depth=25, random_state=42, n_jobs=1)),
        ('et', ExtraTreesClassifier(n_estimators=150, max_depth=25, random_state=43, n_jobs=1))
    ],
    'rf_lgbm': [
        ('rf', RandomForestClassifier(n_estimators=150, max_depth=20, random_state=42, n_jobs=1)),
        ('lgb', LGBMClassifier(n_estimators=100, learning_rate=0.1, random_state=44, n_jobs=1))
    ]
}

meta_models = {
    'logreg': LogisticRegression(max_iter=1000),
    'lgbm': LGBMClassifier(n_estimators=150, learning_rate=0.05, random_state=42, n_jobs=1)
}


# 결과 저장용
results = []

for set_name, base_models in base_model_sets.items():
    for meta_name, meta_model in meta_models.items():
        print(f"\n[Testing] Base={set_name}, Meta={meta_name}")
        stack_model = StackingClassifier(
            estimators=base_models,
            final_estimator=meta_model,
            cv=2,
            n_jobs=1,
            passthrough=True
        )
        pipe = Pipeline([
            ('select', SelectKBest(chi2, k=10000)),
            ('stack', stack_model)
        ])
        scores = cross_val_score(pipe, X_train, y_train, cv=2, scoring='accuracy', n_jobs=1)
        mean_acc = np.mean(scores)
        print(f"Accuracy: {mean_acc:.4f}")
        results.append((set_name, meta_name, mean_acc))

# 결과 DataFrame으로 정리
import pandas as pd
result_df = pd.DataFrame(results, columns=['BaseSet', 'MetaModel', 'Accuracy'])
print("\n최적 조합 결과:")
print(result_df.sort_values(by='Accuracy', ascending=False))



[Testing] Base=rf_et, Meta=logreg
Accuracy: 0.7954

[Testing] Base=rf_et, Meta=lgbm
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.232751 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 36063
[LightGBM] [Info] Number of data points in the train set: 22827, number of used features: 1616
[LightGBM] [Info] Start training from score -2.247488
[LightGBM] [Info] Start training from score -1.993000
[LightGBM] [Info] Start training from score -1.824760
[LightGBM] [Info] Start training from score -2.040393
[LightGBM] [Info] Start training from score -1.789266
[LightGBM] [Info] Start training from score -1.884943
[LightGBM] [Info] Start training from score -1.911252


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.237756 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 37661
[LightGBM] [Info] Number of data points in the train set: 22827, number of used features: 1690
[LightGBM] [Info] Start training from score -2.247488
[LightGBM] [Info] Start training from score -1.993000
[LightGBM] [Info] Start training from score -1.824760
[LightGBM] [Info] Start training from score -2.040730
[LightGBM] [Info] Start training from score -1.789003
[LightGBM] [Info] Start training from score -1.884654
[LightGBM] [Info] Start training from score -1.911549


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Accuracy: 0.7340

[Testing] Base=rf_lgbm, Meta=logreg
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.239399 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 32493
[LightGBM] [Info] Number of data points in the train set: 22827, number of used features: 1602
[LightGBM] [Info] Start training from score -2.247488
[LightGBM] [Info] Start training from score -1.993000
[LightGBM] [Info] Start training from score -1.824760
[LightGBM] [Info] Start training from score -2.040393
[LightGBM] [Info] Start training from score -1.789266
[LightGBM] [Info] Start training from score -1.884943
[LightGBM] [Info] Start training from score -1.911252
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.051343 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_co

...

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.073338 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 13296
[LightGBM] [Info] Number of data points in the train set: 11414, number of used features: 801
[LightGBM] [Info] Start training from score -2.247532
[LightGBM] [Info] Start training from score -1.993365
[LightGBM] [Info] Start training from score -1.824532
[LightGBM] [Info] Start training from score -2.040774
[LightGBM] [Info] Start training from score -1.788785
[LightGBM] [Info] Start training from score -1.884410
[LightGBM] [Info] Start training from score -1.911889


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.257287 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 37661
[LightGBM] [Info] Number of data points in the train set: 22827, number of used features: 1690
[LightGBM] [Info] Start training from score -2.247488
[LightGBM] [Info] Start training from score -1.993000
[LightGBM] [Info] Start training from score -1.824760
[LightGBM] [Info] Start training from score -2.040730
[LightGBM] [Info] Start training from score -1.789003
[LightGBM] [Info] Start training from score -1.884654
[LightGBM] [Info] Start training from score -1.911549


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Accuracy: 0.7322

최적 조합 결과:
   BaseSet MetaModel  Accuracy
0    rf_et    logreg  0.795418
2  rf_lgbm    logreg  0.786831
1    rf_et      lgbm  0.734043
3  rf_lgbm      lgbm  0.732225


## 최적 모델로 스태킹 적용

In [ ]:
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.pipeline import Pipeline
import pandas as pd

base_estimators = [
    ('rf', RandomForestClassifier(
        n_estimators=300,
        max_depth=30,
        min_samples_split=3,
        min_samples_leaf=1,
        max_features='sqrt',
        random_state=42,
        n_jobs=1
    )),

    ('et', ExtraTreesClassifier(
        n_estimators=300,
        max_depth=30,
        min_samples_split=3,
        min_samples_leaf=1,
        max_features='sqrt',
        random_state=43,
        n_jobs=1
    ))
]

meta_model = LogisticRegression(
    max_iter=2000,
    solver='lbfgs',
    multi_class='multinomial',
    n_jobs=-1
)

from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.model_selection import cross_val_score

stack_pipeline = Pipeline([
    ('select', SelectKBest(chi2, k=8000)),
    ('stack', StackingClassifier(
        estimators=base_estimators,
        final_estimator=meta_model,
        cv=3,
        n_jobs=1,
        passthrough=True
    ))
])

from sklearn.model_selection import cross_val_score
import numpy as np

scores = cross_val_score(stack_pipeline, X_train, y_train, cv=3, scoring='f1_macro', n_jobs=1)
print(f"Cross-val F1_macro mean: {scores.mean():.4f} (+/- {scores.std():.4f})")

stack_pipeline.fit(X_train, y_train)
stack_pred = stack_pipeline.predict(X_test)

sample_submission['topic_idx'] = stack_pred
output_path = '/content/drive/MyDrive/data/뉴스토픽_최종스태킹_RFET_LogReg.csv'
sample_submission.to_csv(output_path, index=False)
print(f"제출 파일 저장 완료: {output_path}")

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Cross-val F1_macro mean: 0.8050 (+/- 0.0259)


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


제출 파일 저장 완료: /content/drive/MyDrive/data/뉴스토픽_최종스태킹_RFET_LogReg.csv
